# 05 · Dataset accessor — quiver & compare

`ds.tp` adds Dataset-level views that need more than one variable:

- `ds.tp.quiver_html(...)` — vector arrows (`u`, `v`) over an optional scalar `background`.
- `ds.tp.compare_html(...)` — two variables side-by-side on a shared color scale (model vs reanalysis).

Both write self-contained 2D-map HTML (they default to `equirectangular`).

In [ ]:
import numpy as np
import xarray as xr
import pyterraplot
from pathlib import Path
from IPython.display import IFrame

# Build a small wind-ish dataset: u, v components + speed background
nlat, nlon = 49, 96
lats = np.linspace(80, -80, nlat)
lons = np.linspace(-180, 180, nlon)
LON, LAT = np.meshgrid(lons, lats)

u = (-np.sin(np.radians(LAT)) * np.cos(np.radians(LON))).astype("float32") * 12
v = ( np.cos(np.radians(2*LAT)) * np.sin(np.radians(LON))).astype("float32") * 12
speed = np.sqrt(u**2 + v**2)

ds = xr.Dataset(
    {
        "u10":   (["lat", "lon"], u,    {"units": "m/s", "long_name": "10m u-wind"}),
        "v10":   (["lat", "lon"], v,    {"units": "m/s", "long_name": "10m v-wind"}),
        "speed": (["lat", "lon"], speed,{"units": "m/s", "long_name": "wind speed"}),
    },
    coords={"lat": lats, "lon": lons},
)
ds

## `quiver_html` — arrows over a background

`quiver_cmap` colors arrows by magnitude; `quiver_density` subsamples the grid; `quiver_scale` sets pixels per max-magnitude arrow.

In [ ]:
ds.tp.quiver_html("quiver.html", u="u10", v="v10", background="speed",
                  cmap="YlGnBu", quiver_cmap="plasma", quiver_density=3,
                  projection="naturalEarth", title="winds over speed")
print("quiver.html", f"{Path('quiver.html').stat().st_size/1024:.0f} kB")
IFrame("quiver.html", width="100%", height=420)

## `compare_html` — two panels, shared scale

By default `symmetric=True` centers the (diverging) scale on zero. Here we compare `u10` and `v10`.

In [ ]:
ds.tp.compare_html("compare.html", a="u10", b="v10",
                   cmap="RdBu_r", projection="naturalEarth",
                   title="u10 vs v10")
print("compare.html", f"{Path('compare.html').stat().st_size/1024:.0f} kB")
IFrame("compare.html", width="100%", height=420)